# Predictive Maintenance — Interactive Exploration

This notebook walks through the data and features interactively.
Run `main.py` first to download the data.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.data_loader import download_and_load
from src.feature_engineering import build_feature_matrix, FEATURE_NAMES

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
df = download_and_load(max_segments_per_file=30)
df.head()

In [ ]:
# Class distribution
df['label'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Segments per Class')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()

In [ ]:
# Compare kurtosis across classes — the classic early-fault indicator
from src.feature_engineering import kurtosis_val

df['kurtosis'] = df['signal'].apply(kurtosis_val)
df.boxplot(column='kurtosis', by='label', figsize=(8, 4))
plt.suptitle('')
plt.title('Kurtosis by Fault Class')
plt.ylabel('Kurtosis')
plt.xticks(rotation=30)
plt.tight_layout()

In [ ]:
# Build full feature matrix
X, y, feat_names, label_map = build_feature_matrix(df)
print('Feature matrix:', X.shape)
print('Classes:', label_map)

In [ ]:
# PCA visualisation — do features separate the classes?
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Xs = StandardScaler().fit_transform(X)
pcs = PCA(n_components=2).fit_transform(Xs)

colors = ['#2196F3', '#F44336', '#FF9800', '#4CAF50']
inv_map = {v: k for k, v in label_map.items()}

fig, ax = plt.subplots(figsize=(8, 6))
for cls_idx, color in zip(sorted(inv_map), colors):
    mask = y == cls_idx
    ax.scatter(pcs[mask, 0], pcs[mask, 1], s=15, alpha=0.6,
               color=color, label=inv_map[cls_idx])
ax.legend()
ax.set_title('PCA of Extracted Features (2 components)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout()